In [ ]:
import pandas as pd
import numpy as np
from sklearn import model_selection

In [ ]:
# Used To Format Questions
class Question:
    def __init__(self, column, value, root=pd.DataFrame, information_gain=float, truthTable=pd.DataFrame, falseTable=pd.DataFrame, truthQuestion=None, falseQuestion=None):
        self.column = column
        self.value = value

        self.depth = 0
        self.information_gain = information_gain
        
        self.root = root
        self.truthTable = truthTable
        self.falseTable = falseTable

        self.truthQuestion = truthQuestion
        self.falseQuestion = falseQuestion
    
    # Checks if x makes question True or False *used only in prediction*
    def result(self, x=pd.DataFrame):

        if (isinstance(self.value, (int, np.integer, np.floating))):
            if x[x.columns[self.column]][0] >= self.value:
                return self.truthQuestion, self.truthTable["Group"]
            else:
                return self.falseQuestion, self.falseTable["Group"]

        else:
            if x[x.columns[self.column]][0] == self.value:
                return self.truthQuestion, self.truthTable["Group"]
            else:
                return self.falseQuestion, self.falseTable["Group"]


In [ ]:
class Tree:
    def __init__(self, df):
        self.root = df
        self.questionTree = []
        self.question = None

        self.max_depth = 10

        self.grow_tree(self.root)
    
    # Generates the best question to ask at every node
    def grow_tree(self, root):
        best_question = self.find_best_question(root)


        if best_question != -1:
            self.questionTree.append(best_question)

            best_question.truthQuestion = self.grow_tree(best_question.truthTable)

            best_question.falseQuestion = self.grow_tree(best_question.falseTable) 
        
            return self.questionTree.index(best_question)
        self.question = self.questionTree[0] 

    # Splits a root based on a question into True Vales, and False Values

    def partition(self, rows=pd.DataFrame, question=Question):

        if (isinstance(question.value, (int, np.integer, np.floating))) == False:
            true_values = rows[rows[rows.columns[question.column]] == question.value]    
            false_values = rows[rows[rows.columns[question.column]] != question.value]
        else:
            true_values = rows[rows[rows.columns[question.column]] >= question.value]    
            false_values = rows[rows[rows.columns[question.column]] < question.value]

        return true_values, false_values

    # Calculate Impurity
    def impurity(self, rows=pd.DataFrame):
        columnLen = len(rows["Group"])
        impurity = 1
        for i in rows["Group"].unique():
            impurity -= ((rows["Group"].value_counts()[i]) / columnLen)**2
        return impurity

    # Check how good a question is by comparing it to the previous node
    def information_gain(self, rootTable=pd.DataFrame, trueTable=pd.DataFrame, falseTable=pd.DataFrame, question=Question):
        rootImpurity = self.impurity(rootTable)

        trueImpurity = self.impurity(trueTable)
        falseImpurity = self.impurity(falseTable)

        AvgImpurity = (trueImpurity * (trueTable.shape[0] / rootTable.shape[0])) + (falseImpurity * (falseTable.shape[0] / rootTable.shape[0])) 
        AvgImpurity /= 2

        return rootImpurity - AvgImpurity

    # Combines Last 3 to find the best question at a single node
    def find_best_question(self, root=pd.DataFrame):
        best_question = -1
        best_information_gain = -1
        pdroot = root
        for column in range(0, len(root.drop(columns=["Group"]).columns)):
            for value in root[root.columns[column]].unique():
                question = Question(column, value, root=pdroot)

                question.truthTable, question.falseTable = self.partition(root, question)

                if question.truthTable.shape[0] == 0 or question.falseTable.shape[0] == 0:
                    continue

                question.information_gain = self.information_gain(root, question.truthTable, question.falseTable, question)
                
                if best_information_gain == None or question.information_gain > best_information_gain:
                    # print("Is ", question.column, " equal too", question.value)
                    best_information_gain = question.information_gain
                    best_question = question

        if len(self.questionTree) != 0:
            if type(best_question) == Question:
                    for i in self.questionTree:
                        if i.truthTable.equals(root) or i.falseTable.equals(root):
                            best_question.depth = i.depth + 1
                            if best_question.depth > self.max_depth:
                                print("THAT RIGHT THERE WAS TOO MUCH, GET IT GONNEN")
                                return -1
            else:
                return -1

        if best_information_gain == 0:
            return -1
        return best_question
    
    # Just to help visualize
    def tree_ToString(self):
        for pos, q in enumerate(self.questionTree):
            print("Question #",pos, ": ", self.root.columns[q.column], " is ", q.value, " ----> Question #", q.truthQuestion, "or Question #", q.falseQuestion)
    
    # Uses the treeQuestions list to predict a x row
    def predict(self, x):
        global prediction
        # print(self.questionTree.index(self.question))
        if self.question.result(x)[0] != None:
               self.question = self.questionTree[self.question.result(x)[0]]
               self.predict(x)
        else:
            prediction = self.question.result(x)[1].value_counts().sort_values(ascending=False).index[0]
        self.question = self.questionTree[0]
             

In [ ]:
df = pd.read_csv("alzheimer.csv")

In [ ]:
df["M/F"].value_counts().sort_values(ascending=False).index[0]

In [ ]:
df.head(5)

In [ ]:
# CLEANING

In [ ]:
# remove nan
df = df[df["SES"].isin([x for x in range(0, 6)])]

In [ ]:
# Changes Male and Female to ones and zeros (unnecessary but i did it)
df["M/F"].replace("M", 0, inplace=True)
df["M/F"].replace("F", 1, inplace=True)

In [ ]:
# Basically, theres too much nonDemented so we split it...

demented = df[df["Group"] == "Demented"]
nonDemented = df[df["Group"] == "Nondemented"]

demented.reset_index(inplace=True, drop=True)
nonDemented.reset_index(inplace=True, drop=True)

In [ ]:
difference = nonDemented.shape[0] - demented.shape[0]
difference

In [ ]:
# ... Then splice demented to make it match demented
nonDemented = nonDemented[nonDemented.index.isin([x for x in range(0, nonDemented.shape[0] - difference)])]

In [ ]:
# Represents demented and nondemented ans 1, and 0 
demented.replace("Demented", 1, inplace=True)
nonDemented.replace("Nondemented", 0, inplace=True)

In [ ]:
# Rejoins the split
df = pd.concat([demented, nonDemented], axis=0)

In [ ]:
# Cleanup
df.reset_index(inplace=True)
df.drop(columns=["index"], inplace=True)

In [ ]:
# Including CDR makes the bot perfect (cuz it becomes obvious), so we dont use it
df.drop(columns=["CDR"], inplace=True)

In [ ]:
# MAKING TRAINING DATA

In [ ]:
# features are the things we feed to the bot
feature = df.drop(columns=["Group"])

# classifiers are the answers
classifier = df["Group"]

In [ ]:
# Separates the data into training data (given to bot with answers), and testing data (given to bot without answers to test accuracy)
x_train, x_test, y_train, y_test = model_selection.train_test_split(feature, classifier, test_size=0.2)

In [ ]:
# merge training data
x_train["Group"] = y_train
x_train

In [ ]:
# Creates the Model
tree = Tree(x_train)

In [ ]:
# Visualization (if it leads to None, then we can predict from there)
tree.tree_ToString()

In [ ]:
# Prepares testing data to be assessed by the bot
x_test.reset_index(inplace=True, drop=True)
y_test.reset_index(inplace=True, drop=True)

In [ ]:
# Tests the bot
accuracy = 0
for qNumber in range(x_test.shape[0]):
    row = x_test[x_test.index == qNumber]
    row.reset_index(inplace=True, drop=True)
    print(row)
    prediction = None
    tree.predict(row)
    print(prediction)
    if prediction == y_test[qNumber]:
        accuracy += 1


In [ ]:
# Bot Accuracy
accuracy / x_test.shape[0]